# STEP 2: Fetch High Court Judgment Metadata (Public S3)

Source: `s3://indian-high-court-judgments` (public)

This notebook fetches metadata parquet files (not PDFs), filters rows with description text, and saves results for downstream labeling.

## Notes
- Designed for Google Colab.
- Saves per-shard parquet files first, then combines to reduce memory pressure.
- You can change courts/years in the configuration cell.

In [1]:
# Colab setup
%pip install -q pandas pyarrow

import pandas as pd
from pathlib import Path
import time
import gc

In [2]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


In [3]:
# Configuration
BASE_DIR = Path('/content/drive/MyDrive/MiniProject')
OUTPUT_DIR = BASE_DIR / 'compiled_dataset'
SHARD_DIR = OUTPUT_DIR / 'hc_metadata_parts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

S3_BASE = 'https://indian-high-court-judgments.s3.ap-south-1.amazonaws.com'

# Start focused; expand later if needed.
TARGET_COURTS = {
    '27_1': 'Bombay HC',
    '18_6': 'Delhi HC',
    '9_13': 'Allahabad HC',
    '21_11': 'Karnataka HC',
    '28_2': 'Madras HC',
}
TARGET_YEARS = list(range(2015, 2025))  # 2015-2024

# Polite delay between requests
REQUEST_DELAY_SECONDS = 0.2

In [4]:
import pyarrow.dataset as ds
import pyarrow.fs as pafs
import pyarrow.parquet as pq
from collections import Counter


def fetch_parquet(court_code: str, year: int, verbose: bool = False) -> pd.DataFrame:
    """
    Load one court-year metadata slice from the public S3 parquet dataset.
    Dataset is partitioned (year, court, bench), not a single metadata.parquet file.
    """
    s3 = pafs.S3FileSystem(region='ap-south-1', anonymous=True)
    partition_path = f"indian-high-court-judgments/metadata/parquet/year={year}/court={court_code}"

    try:
        dataset = ds.dataset(partition_path, filesystem=s3, format='parquet')
        table = dataset.to_table()
        if table.num_rows == 0:
            return pd.DataFrame()

        df = table.to_pandas()
        df['source_court_code'] = court_code
        df['source_year'] = year
        return df
    except Exception as e:
        if verbose:
            print(f"error: {type(e).__name__}: {e}")
        return pd.DataFrame()


def _parse_mixed_date(series: pd.Series) -> pd.Series:
    """Parse mixed date strings while handling dd-mm-yyyy safely."""
    s = series.astype('string').str.strip()

    # First pass: common Indian date format (day-first).
    parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)

    # Fallback for unresolved ISO-style values.
    missing = parsed.isna() & s.notna() & (s != '')
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            s.loc[missing], errors='coerce', format='%Y-%m-%d'
        )

    return parsed


def prepare_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Parse dates if present (avoid dayfirst warnings)
    for datecol in ['decision_date', 'date_of_registration']:
        if datecol in df.columns:
            df[datecol] = _parse_mixed_date(df[datecol])

    # Keep rows with description text for NLP
    if 'description' in df.columns:
        df = df[df['description'].notna()].copy()

    # Add readable court label
    df['court_name_clean'] = df['source_court_code'].map(TARGET_COURTS)

    return df


def fetch_and_save_parts():
    """Fetch each court-year dataset and save as separate parquet parts."""
    print('=' * 60)
    print('High Court S3 Metadata Fetcher (Part Writer)')
    print('=' * 60)
    print(f"Target courts: {list(TARGET_COURTS.values())}")
    print(f"Target years:  {TARGET_YEARS}")
    print(f"Total requests to attempt: {len(TARGET_COURTS) * len(TARGET_YEARS)}\n")

    written_parts = []
    total_rows = 0

    for court_code, court_name in TARGET_COURTS.items():
        for year in TARGET_YEARS:
            print(f"  Fetching {court_name} ({court_code}) - {year} ...", end=' ', flush=True)
            # verbose=True so failures show the real reason
            df = fetch_parquet(court_code, year, verbose=True)
            if df.empty:
                print('no data')
            else:
                df = prepare_columns(df)
                if df.empty:
                    print('0 rows after filtering')
                else:
                    part_path = SHARD_DIR / f"hc_{court_code}_{year}.parquet"
                    df.to_parquet(part_path, index=False)
                    written_parts.append(part_path)
                    total_rows += len(df)
                    print(f"{len(df):,} rows -> {part_path.name}")

                del df
                gc.collect()

            time.sleep(REQUEST_DELAY_SECONDS)

    print(f"\nPart files written: {len(written_parts)}")
    print(f"Total kept rows: {total_rows:,}")
    return written_parts


def combine_parts_to_final(parts):
    """Combine parquet parts into one output file using streaming writes (low RAM)."""
    if not parts:
        print('\n[ERROR] No part files to combine.')
        return None

    print(f"\n[Combining] {len(parts)} part files (streaming mode) ...")

    out_path = OUTPUT_DIR / 'hc_metadata.parquet'
    writer = None
    total_rows = 0
    court_counter = Counter()
    disposal_counter = Counter()

    for i, p in enumerate(sorted(parts), start=1):
        table = pq.read_table(p)

        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)

        writer.write_table(table)
        total_rows += table.num_rows

        # Lightweight summary from each part
        df_small = table.to_pandas()
        if 'court_name_clean' in df_small.columns:
            court_counter.update(df_small['court_name_clean'].dropna().astype(str).tolist())
        if 'disposal_nature' in df_small.columns:
            disposal_counter.update(df_small['disposal_nature'].dropna().astype(str).tolist())

        del df_small
        del table
        gc.collect()

        if i % 10 == 0 or i == len(parts):
            print(f"  merged {i}/{len(parts)} parts ...")

    if writer is not None:
        writer.close()

    print(f"Saved -> {out_path}")
    print(f"Rows: {total_rows:,}")
    print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

    print('\n-- Summary --')
    if court_counter:
        print('\nRows per court:')
        print(pd.Series(court_counter).sort_values(ascending=False))

    if disposal_counter:
        print('\nTop disposal natures:')
        print(pd.Series(disposal_counter).sort_values(ascending=False).head(15))

    gc.collect()
    return out_path

In [5]:
# Run Step 2
parts = fetch_and_save_parts()
final_path = combine_parts_to_final(parts)
print('\nDone')

High Court S3 Metadata Fetcher (Part Writer)
Target courts: ['Bombay HC', 'Delhi HC', 'Allahabad HC', 'Karnataka HC', 'Madras HC']
Target years:  [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Total requests to attempt: 50

  Fetching Bombay HC (27_1) - 2015 ... 99,150 rows -> hc_27_1_2015.parquet
  Fetching Bombay HC (27_1) - 2016 ... 

/tmp/ipykernel_22762/2721385828.py:36: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)


100,742 rows -> hc_27_1_2016.parquet
  Fetching Bombay HC (27_1) - 2017 ... 

/tmp/ipykernel_22762/2721385828.py:36: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)


102,153 rows -> hc_27_1_2017.parquet
  Fetching Bombay HC (27_1) - 2018 ... 

...

29,781 rows -> hc_28_2_2021.parquet
  Fetching Madras HC (28_2) - 2022 ... 

/tmp/ipykernel_22762/2721385828.py:36: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)


46,051 rows -> hc_28_2_2022.parquet
  Fetching Madras HC (28_2) - 2023 ... 

/tmp/ipykernel_22762/2721385828.py:36: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed = pd.to_datetime(s, errors='coerce', dayfirst=True)


49,775 rows -> hc_28_2_2023.parquet
  Fetching Madras HC (28_2) - 2024 ... 57,419 rows -> hc_28_2_2024.parquet

Part files written: 50
Total kept rows: 3,433,684

[Combining] 50 part files (streaming mode) ...
  merged 10/50 parts ...
  merged 20/50 parts ...
  merged 30/50 parts ...
  merged 40/50 parts ...
  merged 50/50 parts ...
Saved -> /content/drive/MyDrive/MiniProject/compiled_dataset/hc_metadata.parquet
Rows: 3,433,684
File size: 1481.9 MB

-- Summary --

Rows per court:
Allahabad HC    1428780
Bombay HC        909491
Karnataka HC     667576
Madras HC        221190
Delhi HC         206647
dtype: int64

Top disposal natures:
Disposed Off                                       696177
Disposed off/Decided on merits                     509843
Allowed/Partly Allowed on merits                   360096
Dismissed on merits                                226824
Dismiss other than merit(DD/Non Prosec./Abated)    187678
DISPOSED OFF                                       158000
Disposed Of

In [6]:
# Copy fetched dataset to a dedicated Drive folder
import shutil

TARGET_DIR = Path('/content/drive/MyDrive/MiniProject/IHCJ_dataset_180426')
TARGET_DIR.mkdir(parents=True, exist_ok=True)

# Prefer combined file if available
combined_file = OUTPUT_DIR / 'hc_metadata.parquet'
if combined_file.exists():
    shutil.copy2(combined_file, TARGET_DIR / combined_file.name)
    print(f'Copied: {combined_file.name} -> {TARGET_DIR}')
else:
    print('Combined file not found; skipping hc_metadata.parquet copy')

# Also copy part files folder for safety/reproducibility
parts_target = TARGET_DIR / 'hc_metadata_parts'
if SHARD_DIR.exists():
    shutil.copytree(SHARD_DIR, parts_target, dirs_exist_ok=True)
    print(f'Copied parts folder -> {parts_target}')
else:
    print('Parts folder not found; nothing to copy')

print('Upload to Drive folder IHCJ_dataset_180426 completed')

Copied: hc_metadata.parquet -> /content/drive/MyDrive/MiniProject/IHCJ_dataset_180426
Copied parts folder -> /content/drive/MyDrive/MiniProject/IHCJ_dataset_180426/hc_metadata_parts
Upload to Drive folder IHCJ_dataset_180426 completed
